In [6]:
import numpy as np
import pandas as pd
import kagglehub
import pathlib
from pathlib import Path
import cv2
from collections import Counter
# from src.config import (
#       DATASET_CACHE_PATH,
#       DATASET_SLUG,
#       LABELS_BY_FOLDER,
#       VALID_EXTENSIONS,
# )
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split

DATASET_CACHE_PATH = Path(
      "/root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data"
)
DATASET_SLUG = "devana2004/dataset"

def charger_dataset_path():
      """Retourne le dataset en cache ou le télécharge avec kagglehub."""
      dataset_path = DATASET_CACHE_PATH

      if dataset_path.exists() and any(dataset_path.iterdir()):
            print("Dataset déjà présent dans le cache.")
            print("Chemin :", dataset_path)
            print("Contenu :", [p.name for p in dataset_path.iterdir()])
      else:
            print("Dataset absent ou dossier vide.")
            dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG)) / "Data"

      print("Path :", dataset_path)
      return Path(dataset_path)


def check_data(path, extensions_valides, labels_par_dossier):
      images = []
      labels = []

      num_skipped = 0
      num_corrompu = 0

      data_dir = Path(path)
      extensions_valides = {
            ext.lower() if ext.startswith(".") else f".{ext.lower()}"
            for ext in extensions_valides
      }

      for folder in data_dir.iterdir():
            if not folder.is_dir():
                  continue

            # Ignorer les dossiers qui ne correspondent pas aux classes
            if folder.name not in labels_par_dossier:
                  continue

            for file in folder.iterdir():
                  if not file.is_file():
                        continue

                  # Vérifier l'extension avant de décoder l'image
                  if file.suffix.lower() not in extensions_valides:
                        num_skipped += 1
                        continue

                  image = cv2.imread(str(file))

                  if image is None:
                        num_corrompu += 1
                        continue

                  # BGR -> RGB
                  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                  # Redimensionnement pour MobileNetV2
                  image = cv2.resize(image, (224, 224))

                  images.append(image)
                  labels.append(labels_par_dossier[folder.name])

      print(f"Fichiers ignorés : {num_skipped}")
      print(f"Images corrompues : {num_corrompu}")
      print(f"Images chargées : {len(images)}")
      print(f"Labels chargés : {len(labels)}")

      return images, labels


def preparing_data(images, labels,image_size=(224, 224)):

      images_resized =[]
      
      for image in images:
            resized = cv2.resize(image, image_size, interpolation=cv2.INTER_LINEAR)
            images_resized.append(resized)
      
      X = np.asarray(images_resized, dtype=np.uint8)
      
      y = np.asarray(labels, dtype=np.int32)

      if len(X) != len(y):
            raise ValueError(
                  "Le nombre d'images doit correspondre au nombre de labels."
            )
      
      return X, y

def spliting_data(X, y, random_state=42):

      X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, stratify=y, random_state=random_state)

      X_test, X_val, y_test, y_val = train_test_split(X_tmp, y_tmp, test_size=0.5,  stratify=y_tmp, random_state=random_state)

      print(X_train.shape, X_test.shape, X_val.shape)

      print(y_train.shape, y_test.shape, y_val.shape)

      return  X_train, X_test, X_val, y_train, y_test, y_val


def  normalisation(X, seuil):
      return X.astype('float32') / seuil
LABELS_BY_FOLDER = {
      "glioma": 0,
      "meningioma": 1,
      "notumor": 2,
      "pituitary": 3,
}

CLASS_NAMES = [
      "Gliome",
      "Méningiome",
      "Aucune tumeur",
      "Tumeur hypophysaire",
]

def preprocess(image,label, cnn=True):
      # image = resize(image,image_size)
      image = tf.cast(image, tf.float32)

      # Prétraitement officiel de MobileNetV2 : pixels vers [-1, 1]
      if cnn:
            image = tf.cast(image, tf.float32) / 255.0
      else:
            image = preprocess_input(image)

      label = tf.cast(label, tf.int32)

      return image, label

def prep(X, y, training=True, random_state=42, batch_size=32):

      X_ds = tf.data.Dataset.from_tensor_slices((X, y))

      if training:
            X_ds = X_ds.shuffle(buffer_size=len(X), seed=random_state, reshuffle_each_iteration=True)

      X_ds = X_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
      
      X_ds = X_ds.batch(batch_size=batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

      return X_ds

def print_class_counts(labels, labels_par_dossier=LABELS_BY_FOLDER):
      effectifs = Counter(labels)
      noms_classes = list(labels_par_dossier.keys())
      nombres_images = [
            effectifs[labels_par_dossier[nom]] for nom in noms_classes
      ]

      for nom, nombre in zip(noms_classes, nombres_images):
            print(f"{nom} : {nombre} images")

      return noms_classes, nombres_images



In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from pathlib import Path
import time


def build_data_augmentation():
      return tf.keras.Sequential([
            layers.RandomRotation(0.05),
            layers.RandomZoom(
                  height_factor=(-0.2, 0.2),
                  width_factor=(-0.2, 0.2)
            ),
            layers.RandomTranslation(
                  height_factor=0.05,
                  width_factor=0.05
            ),
            layers.RandomZoom(0.1)
            ], name="data_augmentation")

def  build_cnn():

      """Construit le CNN de base comme dans le notebook."""
      data_augmentation= build_data_augmentation()
      # API Sequentiel
      model = models.Sequential()
      model.add(layers.Input(shape=(224, 224, 3)))
      model.add(data_augmentation)
      model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
      model.add(layers.MaxPooling2D((2, 2)))
      model.add(layers.Conv2D(64, (3, 3), activation='relu'))
      model.add(layers.MaxPooling2D((2, 2)))
      model.add(layers.Conv2D(64, (3, 3), activation='relu'))
      model.add(layers.Flatten())
      model.add(layers.Dense(64, activation='relu'))
      model.add(layers.Dense(4))

      return model

def build_model(image_size=(224, 224),dropout_rate=0.2, num_classes=4):

      data_augmentation= build_data_augmentation()

      base_model = tf.keras.applications.MobileNetV2(
            input_shape=image_size + (3,), include_top=False,
            weights='imagenet'
      )

      #Transfert_Learning : geler le backbone
      base_model.trainable = False

      inputs = tf.keras.Input(shape=(image_size +(3,)))
      x = data_augmentation(inputs)

      # Les images sont déjà prétraitées dans Train_ds
      x = base_model(x, training=False)

      x = layers.GlobalAveragePooling2D()(x)

      x= tf.keras.layers.Dropout(0.2)(x)

      #Sortie en probabilites pour les 4 classes
      outputs = layers.Dense(num_classes, activation='softmax')(x)

      model = tf.keras.Model(inputs, outputs)

      return model, base_model

def compile_mode(model, learning_rate=1e-5):
      model.compile(
            optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
            metrics=['accuracy']
      )

      return model

def evaluate_initial_model(model, val_ds):
      loss, accuracy = model.evaluate(val_ds, verbose=1)

      print(f"Loss initiale : {loss:.4f}")
      print(f"Accuracu initiale : {accuracy:.4f}")

      return {
            "initial_val_loss": float(loss),
            "initial_val_accuracy": float(accuracy)
      }

def create_callbacks(output_dir, backup_dir):
      output_dir = Path(output_dir)
      backup_dir = Path(backup_dir)

      output_dir.mkdir(parents=True, exist_ok=True)
      backup_dir.mkdir(parents=True, exist_ok=True)

      checkpoint_best = tf.keras.callbacks.ModelCheckpoint(
            filepath=str(output_dir / 'best_model.keras'),
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            verbose=1
      )

      early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=4,
            restore_best_weights=True,
            verbose=1
      )

      backup = tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq='epoch',
            double_checkpoint=True,
            delete_checkpoint=False
      )

      return [checkpoint_best, early_stopping, backup]

def train_model(model, train_ds, val_ds, epochs, callbacks):

      debut = time.time()

      history = model.fit(
            train_ds,
            epochs=epochs,
            validation_data=val_ds,
            callbacks=callbacks
      )

      duree = time.time() - debut

      print(f"Durée : {duree:.2f} secondes")
      print(f"Durée : {duree /60:.2f} minutes")

      return (history, duree)


def degler_layers(base_model, num_layers=30):

      #Degeler le model pouir permettre le fine-tuning
      base_model.trainable = True

      #Garder les premiers courches gelées
      for layer in base_model.layers[:-num_layers]:
            layer.trainable = False

      return base_model

In [8]:
RAMDOM_STATE = 42
BATCH_SIZE= 32
EPOCHS = 20
image_size = (224, 224)

extensions_valides = (".jpg", ".jpeg", ".png", ".bmp")

labels_par_dossier = {
    "glioma": 0,
    "meningioma": 1,
    "notumor" : 2,
    "pituitary" : 3
}
dataset_path = Path(
      "/root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data"
)

dataset_path = charger_dataset_path()

images, labels = check_data(dataset_path, extensions_valides,  labels_par_dossier)


images_array_2d, labeles_array_2d = preparing_data(images=images, labels=labels,image_size=image_size)

X_train, X_test, X_val, y_train, y_test, y_val = spliting_data(images_array_2d, labeles_array_2d, random_state=RAMDOM_STATE)

Train_ds =  prep(X_train, y_train, training=True)
Test_ds = prep(X_test, y_test, training=False)
Val_ds = prep(X_val, y_val, training=False)
print(Train_ds, Test_ds, Val_ds)


OUTPUT_DIR = "/cnn/models"
BACKUP_DIR = "/cnn/backup"

# 1. Construire le modèle
model = build_cnn()

# 2. Compiler
model = compile_mode(model, learning_rate=0.0001)

# 3. Vérifier l'architecture
model.summary()

# 4. Évaluer avant entraînement
# initial_metrics = evaluate_initial_model(model, Val_ds)

# 5. Créer les callbacks
callbacks = create_callbacks(OUTPUT_DIR, BACKUP_DIR)

# 6. Entraîner
history, training_duration = train_model(
      model,
      Train_ds,
      Val_ds,
      epochs=EPOCHS,
      callbacks=callbacks
)




Dataset déjà présent dans le cache.
Chemin : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data
Contenu : ['glioma', 'meningioma', 'notumor', 'pituitary']
Path : /root/.cache/kagglehub/datasets/devana2004/dataset/versions/1/Data
Fichiers ignorés : 0
Images corrompues : 0
Images chargées : 7023
Labels chargés : 7023
Fichiers ignorés : 0
Images corrompues : 0
Images chargées : 7023
Labels chargés : 7023
(5618, 224, 224, 3) (702, 224, 224, 3) (703, 224, 224, 3)
(5618,) (702,) (703,)


I0000 00:00:1792869362.287692   22433 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 107 MB memory:  -> device: 0, name: NVIDIA TITAN Xp, pci bus id: 0000:06:00.0, compute capability: 6.1
I0000 00:00:1792869362.288362   22433 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 11382 MB memory:  -> device: 1, name: NVIDIA TITAN Xp, pci bus id: 0000:0b:00.0, compute capability: 6.1
W0000 00:00:1792869362.290741   22433 cpu_allocator_impl.cc:82] Allocation of 845666304 exceeds 10% of free system memory.
W0000 00:00:1792869372.712814   22433 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 806.49MiB (rounded to 845666304)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1792869372.712912   22433 

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

def plot_training_history(history, output_dir):
      output_dir = Path(output_dir)
      output_dir.mkdir(parents=True, exist_ok=True)

      history_data = history.history

      #Courbe d'accuracy

      plt.figure(figsize=(8,5))
      plt.plot(history_data['accuracy'], label ='Train accuracy')

      if 'val_accuracy' in history_data:
            plt.plot(history_data['val_accuracy'], label ='Validation accuracy')

      plt.title('Accuracy - Fine Tuning MobileNetV2')
      plt.xlabel('Epoque')
      plt.ylabel('Accuracy')
      plt.legend()
      plt.grid(True)
      plt.tight_layout()
      plt.savefig(output_dir / "accuracy.png", dpi=150)
      plt.show()
      plt.close()


      #Courbe de loss
      plt.figure(figsize=(8,5))
      plt.plot(history_data['loss'], label ='Train loss')
      
      if 'val_loss' in history_data:
            plt.plot(history_data['val_loss'], label ='Validation loss')

      plt.title('Loss - Fine Tuning MobileNetV2')
      plt.xlabel('Epoque')
      plt.ylabel('Loss')
      plt.legend()
      plt.grid(True)
      plt.tight_layout()
      plt.savefig(output_dir / "loss.png", dpi=150)
      plt.show()
      plt.close()

######
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)


def evaluate_model(model, test_ds, class_names, output_dir):
      output_dir = Path(output_dir)
      output_dir.mkdir(parents=True, exist_ok=True)

      #1 Recuperer les vrais labels
      y_true = np.concatenate([
            labels.numpy() for _, labels in test_ds
      ])

      # 2 Recuperer les probabilites

      y_prob = model.predict(test_ds, verbose=1)

      # 3 Transformer les probalibites en classes predites

      y_pred = np.argmax(y_prob, axis=1)

      # Verifier la correspondance entre les classes et les indices

      class_labels = list(range(len(class_names)))

      # 4 Calculer les metriques globals

      metrics = {
            "accuracy" : float(accuracy_score(y_true=y_true, y_pred=y_pred)),
            "precision_macro" : float(
                  precision_score(
                        y_true=y_true,
                        y_pred=y_pred,labels=class_labels, average='macro', zero_division=0)),

            "recall_macro" : float(
            precision_score(
                  y_true=y_true,
                  y_pred=y_pred,
                  labels=class_labels,
                  average='macro',
                  zero_division=0)),
            "f1_weighted" : float(
            f1_score(
                  y_true=y_true,
                  y_pred=y_pred,
                  labels=class_labels,
                  average='weighted',
                  zero_division=0)),
      }

      # 5 Rapport detailler par class

      raport = classification_report(
            y_true=y_true,
            y_pred=y_pred,
            labels=class_labels,
            target_names=class_names,
            output_dict=True,
            zero_division=0
      )

      print("\n=== Métriques globales ===")
      for name, value in metrics.items():
            print(f"{name}: {value:.4f}")

      print("\n=== Rapport par classe ===")
      print(
            classification_report(
                  y_true,
                  y_pred,
                  labels=class_labels,
                  target_names=class_names,
                  zero_division=0
            )
      )

      # 6 Metrice de confusion
      cm= confusion_matrix(
            y_true=y_true,
            y_pred=y_pred,
            labels=class_labels
      )

      display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

      fig, ax = plt.subplots(figsize=(8,7))
      display.plot(
            ax=ax,
            cmap='Blues',
            values_format='d',
            colorbar=False
      )

      ax.set_title("Matrice de confusion - CNN")
      fig.tight_layout()
      fig.savefig(
            output_dir / "confusion_matrix.png", dpi=150,
            bbox_inches='tight'
      )
      plt.show()
      plt.close(fig)

      # 7 Sauvegarder les résultats en JSON
      results = {
            "metrics": metrics,
            "classification_report": raport,
            "confusion_matrix": cm.tolist(),
            "class_names": class_names,
            "test_samples": int(len(y_true))
      }

      with open(output_dir / "metrics.json", "w", encoding='utf-8') as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

      print(f"\nResultats sauvgardés dasn : {output_dir}")

      return results

model = tf.keras.models.load_model(
    "/cnn/models/best_model.keras"
) 
labels_par_dossier = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

class_names = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

# from src.evaluation.plots import plot_training_history
# from src.evaluation.metrics import evaluate_model

# Visualisation de l'entraînement
# plot_training_history(
#     history,
#     output_dir="/transfer_learning/artifacts/plots"
# )

# Évaluation finale sur le jeu de test
results = evaluate_model(
    model=model,
    test_ds=Test_ds,
    class_names=class_names,
    output_dir="/cnn/artifacts/metrics"
)


In [ ]:
from pathlib import Path
import mlflow

def log_transfer_learning_experiment(model_path, results, training_duration, epochs, batch_size, learning_rate, artifact_dict=None):
      mlflow.set_tracking_uri('http://127.0.0.1:5000')
      mlflow.set_experiment('NeuroVision_AI')

      with mlflow.start_run(run_name='cnn') as run:

            #Parameteres de l'experience
            mlflow.log_params({
                  "model" : "cnn",
                  "strategy" : "cnn",
                  "image_size" : "224x224",
                  "num_classes" : 4,
                  "epochs_requested" : epochs,
                  "Batch_size" : batch_size,
                  "learning_rate" : learning_rate,
                  "optimizer" : "Adam",
                  "loss" : "SparseCategoricalCrossentropy",
                  "augmentation" : True
            })

            #Metriques globales calculées sur le jeu de test

            for name, value in results['metrics'].items():
                  mlflow.log_metric(name, float(value))

            mlflow.log_metric("training_duration_seconds", float(training_duration))

            # Rapport , matrice de confusion et autres fichiers : 
            if artifact_dict:
                  for directory in artifact_dict:
                        directory = Path(directory)
                        if directory.exists():
                              mlflow.log_artifacts(str(directory))

            # Enregistrer le model existant sans réentrainer

            if Path(model_path).exists():
                  mlflow.log_artifact(str(model_path), artifact_path="model_files")
            else:
                  raise FileNotFoundError(
                        f"Mode introuvable : {model_path}"
                  )
            print("Run Mlflow enregistré : ", run.info.run_id)

log_transfer_learning_experiment(
      model_path="/cnn/models/best_model.keras",
      results=results,
      training_duration=training_duration,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      learning_rate=1e-5,
      artifact_dict=[
            "/cnn/artifacts/plots",
            "/cnn/artifacts/metrics"
      ]
)